# FakeBDTeen Feature Extractor (Notebook 1)
This notebook builds a cached feature store by extracting aligned audio/video embeddings for all videos. Run the cells top-to-bottom once on GPU.

In [ ]:
import os
import cv2
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchvision.transforms as T
from torchvision.models import resnet18
from transformers import Wav2Vec2Model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

audio_encoder = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-xls-r-300m')
audio_encoder.eval()
for p in audio_encoder.parameters():
    p.requires_grad = False
audio_encoder.to(device)

video_encoder = resnet18(weights='IMAGENET1K_V1')
video_encoder.fc = nn.Identity()
video_encoder.eval()
for p in video_encoder.parameters():
    p.requires_grad = False
video_encoder.to(device)

frame_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def compile_fakebdteen_registry(root_dir: str, output_csv: str) -> pd.DataFrame:
    records = []
    for gender in sorted(os.listdir(root_dir)):
        gender_path = os.path.join(root_dir, gender)
        if not os.path.isdir(gender_path):
            continue
        for authenticity_class in sorted(os.listdir(gender_path)):
            class_path = os.path.join(gender_path, authenticity_class)
            if not os.path.isdir(class_path):
                continue
            video_label = 1 if 'Fake_Video' in authenticity_class else 0
            audio_label = 1 if 'Fake_Audio' in authenticity_class else 0
            for language in sorted(os.listdir(class_path)):
                language_path = os.path.join(class_path, language)
                if not os.path.isdir(language_path):
                    continue
                for subject_id in sorted(os.listdir(language_path)):
                    subject_path = os.path.join(language_path, subject_id)
                    if not os.path.isdir(subject_path):
                        continue
                    for file_name in sorted(os.listdir(subject_path)):
                        if not file_name.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                            continue
                        video_path = os.path.join(subject_path, file_name)
                        unique_id = f"{gender}_{authenticity_class}_{language}_{subject_id}_{os.path.splitext(file_name)[0]}"
                        records.append({
                            'unique_id': unique_id,
                            'gender': gender,
                            'authenticity_class': authenticity_class,
                            'language': language,
                            'subject_id': subject_id,
                            'video_path': video_path,
                            'video_label': video_label,
                            'audio_label': audio_label
                        })
    df = pd.DataFrame.from_records(records)
    df.to_csv(output_csv, index=False)
    return df

dataset_root = '/kaggle/input/datasets/tanmoykdas/fakebdteen/FakeBDTeen'
metadata_csv = '/kaggle/working/fakebdteen_metadata.csv'
registry_df = compile_fakebdteen_registry(dataset_root, metadata_csv)
print(registry_df.head())
print(f'Total videos: {len(registry_df)}')

## 1. Dependencies & Encoder Setup
Load frozen XLS-R and ResNet-18 backbones, configure device, and define frame transforms.

In [ ]:
def extract_audio_features(video_path: str, target_timesteps: int = 100) -> np.ndarray:
    waveform, sample_rate = torchaudio.load(video_path)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)
    waveform = waveform.to(device)
    with torch.no_grad():
        audio_out = audio_encoder(waveform).last_hidden_state
        audio_out = audio_out.transpose(1, 2)
        audio_out = F.interpolate(audio_out, size=target_timesteps, mode='linear', align_corners=False)
        audio_out = audio_out.transpose(1, 2).squeeze(0)
    return audio_out.cpu().numpy().astype(np.float32)

def extract_video_features(video_path: str, target_timesteps: int = 100) -> np.ndarray:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f'Failed to open video: {video_path}')
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        raise RuntimeError(f'Empty video: {video_path}')
    indices = np.linspace(0, frame_count - 1, target_timesteps).astype(np.int64)
    frames = []
    current_idx = 0
    target_set = set(indices.tolist())
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if current_idx in target_set:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame_transform(frame))
        current_idx += 1
    cap.release()
    if len(frames) != target_timesteps:
        if len(frames) == 0:
            raise RuntimeError(f'No frames sampled: {video_path}')
        last_frame = frames[-1]
        while len(frames) < target_timesteps:
            frames.append(last_frame)
        frames = frames[:target_timesteps]
    batch = torch.stack(frames, dim=0).to(device)
    with torch.no_grad():
        feats = video_encoder(batch)
    return feats.cpu().numpy().astype(np.float32)

## 2. Dataset Registry & Labels
Scan the dataset directory and create the metadata CSV with multi-task labels.

In [ ]:
from tqdm import tqdm

features_root = '/kaggle/working/features'
audio_root = os.path.join(features_root, 'audio')
video_root = os.path.join(features_root, 'video')
os.makedirs(audio_root, exist_ok=True)
os.makedirs(video_root, exist_ok=True)

for _, row in tqdm(registry_df.iterrows(), total=len(registry_df)):
    unique_id = row['unique_id']
    video_path = row['video_path']
    audio_out_path = os.path.join(audio_root, f'{unique_id}.npy')
    video_out_path = os.path.join(video_root, f'{unique_id}.npy')
    if os.path.exists(audio_out_path) and os.path.exists(video_out_path):
        continue
    audio_features = extract_audio_features(video_path, target_timesteps=100)
    video_features = extract_video_features(video_path, target_timesteps=100)
    np.save(audio_out_path, audio_features, allow_pickle=False)
    np.save(video_out_path, video_features, allow_pickle=False)

## 3. Aligned Feature Extraction
Extract and temporally align audio/video features to 100 steps.

In [ ]:
audio_files = [f for f in os.listdir(audio_root) if f.endswith('.npy')]
video_files = [f for f in os.listdir(video_root) if f.endswith('.npy')]
print(f'Audio features: {len(audio_files)}')
print(f'Video features: {len(video_files)}')

if len(audio_files) > 0:
    sample_audio = np.load(os.path.join(audio_root, audio_files[0]))
    print(f'Sample audio shape: {sample_audio.shape}')
if len(video_files) > 0:
    sample_video = np.load(os.path.join(video_root, video_files[0]))
    print(f'Sample video shape: {sample_video.shape}')

def get_folder_size_mb(folder_path: str) -> float:
    total_bytes = 0
    for root, _, files in os.walk(folder_path):
        for name in files:
            fp = os.path.join(root, name)
            total_bytes += os.path.getsize(fp)
    return total_bytes / (1024 * 1024)

size_mb = get_folder_size_mb(features_root)
print(f'Feature store size: {size_mb:.2f} MB')

zip_path = '/kaggle/working/fakebdteen_extracted_features.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive('/kaggle/working/fakebdteen_extracted_features', 'zip', features_root)
print(f'Archive created at: {zip_path}')

## 4. Processing Loop & Verification
Generate .npy files for each sample and build the ZIP archive.